# 08 · Flat grids: `mt.geodesicdome.grid.plane.Plane`

`Plane` implements the same `Manifold` interface as `GeodesicDome`, so code written against
`get_neighbours` / `get_neighbours_in_distance` / `get_faces` works with both.

| option | values |
|---|---|
| lattice | `Lattice.Hexagonal` (6 neighbours, odd rows shifted by ½) · `Lattice.Rectilinear` (4 neighbours) |
| topology | `Topology.Plane` (hard borders) · `Topology.Donut` (edges wrap around, i.e. a torus; neighbour search only, faces are not generated across the wrap) |

Script version: [`examples/08_plane_grids.py`](../08_plane_grids.py) · Needs: numpy, matplotlib, ipywidgets (optional: `ipympl` for live, rotatable figures)

<sub>SPDX-License-Identifier: AGPL-3.0-or-later · Copyright (C) 2022-2026 Masahiro Takatsuka. See the NOTICE file for attribution terms.</sub>

## Set-up

In [ ]:
import nb_setup  # makes mt.geodesicdome and examples/dome_utils.py importable from a checkout

WIDGET = nb_setup.setup()  # live figures with ipympl, static images otherwise

In [ ]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display
from matplotlib.collections import PolyCollection
from nb_setup import LiveFigure

from mt.geodesicdome.grid.plane import Lattice, Plane, Topology

COLOURS = ['#c0392b', '#e67e22', '#f1c40f', '#27ae60', '#2980b9', '#8e44ad', '#16a085', '#d35400']

## Building a grid

In [ ]:
grid = Plane(12, 10, Lattice.Hexagonal, Topology.Plane)
xy = grid.get_all_xyz()[:, :2]
k = grid.get_number_of_vertices_per_face()               # 3 = triangles, 4 = quads
faces = grid.get_all_triangles().reshape(-1, k)
print(f'{len(xy)} vertices, {len(faces)} faces with {k} corners each')

v = grid.get_vertex_at(5, 5)
grid.unmark_vertices()
print(f'vertex (5, 5) has {len(grid.get_neighbours(v, False))} neighbours')

## Neighbour rings from a border vertex

In [ ]:
def draw_grid(ax, w, h, lattice, topology, cx, cy, n_rings):
    grid = Plane(w, h, lattice, topology)
    xy = grid.get_all_xyz()[:, :2]
    k = grid.get_number_of_vertices_per_face()
    faces = grid.get_all_triangles().reshape(-1, k)        # triangles or quads

    centre = grid.get_vertex_at(cx, cy)
    grid.unmark_vertices()
    rings = grid.get_neighbours_in_distance(centre, n_rings)

    ax.add_collection(PolyCollection(xy[faces], facecolors='#eef2f7', edgecolors='#9aa5b1', lw=0.6))
    ax.scatter(*xy.T, s=10, c='#9aa5b1')
    ax.scatter(*xy[centre.id], s=90, c=COLOURS[0], zorder=3)
    for r, ring in enumerate(rings, start=1):
        if ring:
            ax.scatter(*xy[[u.id for u in ring]].T, s=50, c=COLOURS[r % len(COLOURS)], zorder=3)
    ax.autoscale(), ax.set_aspect('equal'), ax.set_axis_off()
    ax.set_title(f'{lattice.name} / {topology.name}\nring sizes {[len(r) for r in rings]}')


def ring_legend(fig, n_rings):
    handles = [plt.Line2D([], [], marker='o', ls='', color=COLOURS[i % len(COLOURS)],
                          label='centre' if i == 0 else f'ring {i}') for i in range(n_rings + 1)]
    fig.legend(handles=handles, loc='lower center', ncol=n_rings + 1)


W, H = 12, 10          # use an even height for a hexagonal donut
RINGS = 3
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for row, lattice in enumerate((Lattice.Hexagonal, Lattice.Rectilinear)):
    for col, topology in enumerate((Topology.Plane, Topology.Donut)):
        draw_grid(axes[row, col], W, H, lattice, topology, 0, 4, RINGS)   # centre on the left border
ring_legend(fig, RINGS)
fig.suptitle(f'Plane({W}, {H}): neighbour rings from a border vertex', fontsize=14)
fig.tight_layout(rect=(0, 0.05, 1, 1))
plt.show()

### Try it

Move the centre to a border or a corner and switch between `Plane` and `Donut` to see the rings wrap
around. (Keep the height even for a hexagonal donut, so the row shift lines up across the wrap.)

In [ ]:
w = widgets.IntSlider(value=12, min=3, max=30, description='width')
h = widgets.IntSlider(value=10, min=2, max=30, description='height')
lattice = widgets.ToggleButtons(options=[('Hexagonal', Lattice.Hexagonal), ('Rectilinear', Lattice.Rectilinear)],
                                description='lattice')
topology = widgets.ToggleButtons(options=[('Plane', Topology.Plane), ('Donut', Topology.Donut)],
                                 description='topology')
cx = widgets.IntSlider(value=0, min=0, max=11, description='centre x')
cy = widgets.IntSlider(value=4, min=0, max=9, description='centre y')
n_rings = widgets.IntSlider(value=3, min=1, max=7, description='rings')


def sync(*_):
    cx.max, cy.max = w.value - 1, h.value - 1


w.observe(sync, names='value')
h.observe(sync, names='value')
live = LiveFigure(figsize=(8, 7))


def draw(w, h, lattice, topology, cx, cy, n_rings):
    fig = live.clear()
    draw_grid(fig.add_subplot(), w, h, lattice, topology, min(cx, w - 1), min(cy, h - 1), n_rings)
    ring_legend(fig, n_rings)
    fig.tight_layout(rect=(0, 0.06, 1, 1))
    live.refresh()


out = widgets.interactive_output(draw, dict(w=w, h=h, lattice=lattice, topology=topology,
                                            cx=cx, cy=cy, n_rings=n_rings))
display(widgets.VBox([widgets.HBox([w, h]), lattice, topology, widgets.HBox([cx, cy, n_rings])]), out)
live.show()